# Cellular Automaton: Viral Spread in 2D Tissue

**States per cell:** T · E · V · A · D_V · D_N
**Global scalar:** IFN (felt by all cells, decays over time)

| Transition | Rate / rule |
|---|---|
| T → E | Any virion lands on the cell (deterministic) |
| T → A | `r_a(IFN)` Hill function, zero at IFN = 0 |
| E → V | rate `r_v`, taken with probability `1 - p(IFN)` |
| E → N → D_N | rate `r_v`, taken with probability `p(IFN)`; the N state is instantaneous and releases `dIFN` |
| V → D_V | rate `1/tau_v`, shedding `beta/tau_v` virions per hour while in V |

`p(IFN)` rises from `p_0` (the first-responder probability) toward `p_max` (the
second-responder probability) as IFN accumulates — the paracrine amplification loop.

All timed transitions are single exponential steps, so each state has a geometric
dwell time whose mean is the corresponding time constant.

**Note on dead states.** The SI describes a single dead state D. Here it is split
into `D_V` (died from viral shedding) and `D_N` (died after releasing IFN) purely
for bookkeeping and visualisation. The two are dynamically identical — both inert —
so the split has no effect on results.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from dataclasses import dataclass
import dataclasses
from tqdm import tqdm
import functools
import multiprocess as mp

In [ ]:
@dataclass
class Params:
    """IAV parameter defaults."""

    virus: str = "IAV"   # informational — overridden by subclass

    # Grid
    L:  int   = 250
    dt: float = 0.1

    # Model variant: "basic" | "step_IFN" | "no_IFN" | "strong_IFN"
    mode: str = "basic"

    # E advancement rate; on exit, a Bernoulli trial with p(IFN) picks N vs V.
    r_v:   float = 1/6
    p_0:   float = 5e-3   # first responder probability:  P(N branch) at IFN = 0
    p_max: float = 5e-2   # second responder probability: P(N branch) as IFN -> inf

    IFN_N: float = 10.0   # half-saturation IFN concentration (n_0)

    # Virus spread
    tau_v: float = 6.0
    beta:  float = 100.0
    sigma: float = 1.35 # Whatever gives the right expansion speed

    # Two-phase infectious period: V1 (low shed) → V2 (peak shed) → D_V
    two_V:   bool  = False
    tau_v2:  float = 6.0
    V_ratio: float = 100.0

    # T → A (skipped in step_IFN mode)
    r_a_max: float = 1/10
    IFN_a:   float = None   # half-saturation for r_a; defaults to IFN_N

    dIFN:        float = 1.0
    r_IFN_decay: float = 1/10

    def __post_init__(self):
        # SI Eqs. S7 and S8 share one half-saturation constant n_0. Resolving that
        # here rather than as a class-body default means IFN_a genuinely tracks
        # IFN_N even when IFN_N is overridden on an instance or in a subclass.
        if self.IFN_a is None:
            self.IFN_a = self.IFN_N
        # strong_IFN: same dynamics as "basic", but a stronger IFN response —
        # more IFN released per N-transition, and a higher ceiling on p.
        if self.mode == "strong_IFN":
            self.dIFN  *= 3
            self.p_max *= 4

In [ ]:
def p_of_IFN(IFN: float, par: Params) -> float:
    """P(N branch) at E exit. Equals p_0 at IFN=0, saturates at p_max."""
    if par.mode == "no_IFN":
        return 0.0
    if IFN <= 0:
        return par.p_0
    return par.p_0 + (par.p_max - par.p_0) * IFN / (IFN + par.IFN_N)


def r_a_of_IFN(IFN: float, par: Params) -> float:
    """T -> A rate. Exactly zero at IFN=0, saturates at r_a_max."""
    if IFN <= 0:
        return 0.0
    return par.r_a_max * IFN / (IFN + par.IFN_a)


# ── Quick plot of the two functions ──────────────────────────────────────────
par_demo = Params()
ifn_vals = np.linspace(0, 50, 300)
p_vals   = [p_of_IFN(x, par_demo)  for x in ifn_vals]
ra_vals  = [r_a_of_IFN(x, par_demo) for x in ifn_vals]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ifn_vals, p_vals, 'steelblue')
axes[0].axhline(par_demo.p_max, ls='--', color='red', label=f'p_max = {par_demo.p_max:.3f}')
axes[0].set(title='p (N-branch probability at E exit)', xlabel='IFN', ylabel='probability')
axes[0].legend()

axes[1].plot(ifn_vals, ra_vals, 'darkorange')
axes[1].set(title='T → A rate', xlabel='IFN', ylabel='rate (h⁻¹)')

plt.tight_layout()
plt.show()


In [ ]:
import warnings

class CellularAutomaton:
    """
    State layout (int16 grid):

        0     → T
        1     → E     (eclipse phase)
        2     → V1    (first infectious phase)
        [3]   → V2    (peak shedding; only present if two_V=True)
        next  → A
        next  → D_V
        next  → D_N

    All timed transitions are single exponential steps (rate = 1 / mean duration).

    two_V=True: V1 → V2 → D_V. Shedding rates satisfy r1·tau_v + r2·tau_v2 = beta
                and r2/r1 = V_ratio, so the total expected burst is exactly beta.
    N branch:   an E cell exiting via the N branch (probability p(IFN)) releases
                dIFN instantly and becomes D_N. There is no separate N state, since
                IFN release is treated as immediate.
    step_IFN:   on the step when IFN first appears, all E and V cells immediately
                die to D_V. The T→A transition is skipped in this mode.
    """

    def __init__(self, par: Params, seed=None):
        self.par = par
        self.rng = np.random.default_rng(seed)

        self.T  = 0
        self.E  = 1
        self.V1 = 2

        if par.two_V:
            self.V2  = 3
            nxt      = 4
            self._r1 = par.beta / (par.tau_v + par.V_ratio * par.tau_v2)
            self._r2 = par.V_ratio * self._r1
        else:
            self.V2  = None
            nxt      = 3
            self._r1 = par.beta / par.tau_v
            self._r2 = 0.0

        self.V = np.array([self.V1] if self.V2 is None else [self.V1, self.V2],
                          dtype=np.int16)

        self.A  = nxt
        self.DV = nxt + 1
        self.DN = nxt + 2

        self._front_states = np.array([self.E, *self.V], dtype=np.int16)

        self.grid           = np.zeros((par.L, par.L), dtype=np.int16)
        self.virus          = np.zeros((par.L, par.L), dtype=np.int32)
        self.IFN            = 0.0
        self.t              = 0.0
        self.total_virions  = 0
        self.front_hit_edge = False
        self._center        = np.array([par.L // 2, par.L // 2], dtype=float)

        self._check_dt()

    # ── dt sanity check ───────────────────────────────────────────────────────

    def _check_dt(self):
        par = self.par
        dt  = par.dt
        checks = {
            'T→A (max IFN)': 1 - np.exp(-par.r_a_max * dt),
            'E→V':           1 - np.exp(-par.r_v * dt),
            'V1 advance':    1 - np.exp(-(1 / par.tau_v) * dt),
        }
        if par.two_V:
            checks['V2 advance'] = 1 - np.exp(-(1 / par.tau_v2) * dt)
        bad = {name: prob for name, prob in checks.items() if prob > 0.1}
        if bad:
            lines = '\n'.join(f'  {name}: prob = {prob:.3f}' for name, prob in bad.items())
            warnings.warn(
                f"dt={dt} may be too large — Bernoulli prob > 0.1 for:\n{lines}",
                stacklevel=2,
            )

    # ── Seeding ───────────────────────────────────────────────────────────────

    def seed_infection(self, n: int = 1, pos=None):
        par = self.par
        if pos is None:
            pos = (par.L // 2, par.L // 2)
        for i in range(n):
            self.grid[(pos[0] + i) % par.L, pos[1]] = self.E

    # ── Virus production ──────────────────────────────────────────────────────

    def _deposit(self, state, rate_per_hour):
        par = self.par
        pos = np.argwhere(self.grid == state)
        if len(pos) == 0:
            return
        counts = self.rng.poisson(rate_per_hour * par.dt, size=len(pos))
        total  = int(counts.sum())
        if total == 0:
            return
        self.total_virions += total
        src   = np.repeat(pos, counts, axis=0)
        # Distance ~ half-normal (|N(0, sigma)|), direction uniform on [0, 2*pi).
        r     = np.abs(self.rng.normal(0, par.sigma, total))
        theta = self.rng.uniform(0, 2*np.pi, total)
        dx    = np.round(r * np.cos(theta)).astype(int)
        dy    = np.round(r * np.sin(theta)).astype(int)
        np.add.at(self.virus, ((src[:,0]+dx)%par.L, (src[:,1]+dy)%par.L), 1)

    def _scatter_virions(self):
        self._deposit(self.V1, self._r1)
        if self.par.two_V:
            self._deposit(self.V2, self._r2)

    # ── Main update step ──────────────────────────────────────────────────────

    def step(self):
        par = self.par
        rng = self.rng
        L   = par.L
        dt  = par.dt
        g   = self.grid

        p_ifn = p_of_IFN(self.IFN, par)
        # T→A is skipped in step_IFN mode (no effect there)
        ra    = 0.0 if par.mode == "step_IFN" else r_a_of_IFN(self.IFN, par)

        self.virus[:] = 0
        self._scatter_virions()

        ng      = g.copy()
        new_IFN = self.IFN * np.exp(-par.r_IFN_decay * dt)

        # ── T cells ───────────────────────────────────────────────────────────
        T_mask = g == self.T
        infect = T_mask & (self.virus > 0)
        ng[infect] = self.E
        if ra > 0:
            T_free = T_mask & ~infect
            p_a    = 1 - np.exp(-ra * dt)
            ng[T_free & (rng.random((L, L)) < p_a)] = self.A

        # ── E cells: exit to V1, or via the instantaneous N state to D_N ──────
        p_adv_E   = 1 - np.exp(-par.r_v * dt)
        advancing = (g == self.E) & (rng.random((L, L)) < p_adv_E)
        go_N      = advancing & (rng.random((L, L)) < p_ifn)
        go_V      = advancing & ~go_N
        new_IFN  += int(go_N.sum()) * par.dIFN
        ng[go_N]  = self.DN
        ng[go_V]  = self.V1

        # ── V1: → V2 (if two_V) else D_V ──────────────────────────────────────
        p_adv_V1 = 1 - np.exp(-(1 / par.tau_v) * dt)
        leaving  = (g == self.V1) & (rng.random((L, L)) < p_adv_V1)
        ng[leaving] = self.V2 if par.two_V else self.DV

        # ── V2: → D_V ─────────────────────────────────────────────────────────
        if par.two_V:
            p_adv_V2 = 1 - np.exp(-(1 / par.tau_v2) * dt)
            leaving  = (g == self.V2) & (rng.random((L, L)) < p_adv_V2)
            ng[leaving] = self.DV

        # ── step_IFN: on first IFN signal, kill all E and V ───────────────────
        if par.mode == "step_IFN" and self.IFN <= 0 and new_IFN > 0:
            ng[ng == self.E]        = self.DV
            ng[np.isin(ng, self.V)] = self.DV

        self.grid = ng
        self.IFN  = new_IFN
        self.t   += dt

        # ── Front-reaches-edge check ────────────────────────────────────────
        # With periodic boundaries, once the front (E/V) touches the border
        # it starts wrapping around and colliding with itself.
        border = np.concatenate([ng[0, :], ng[-1, :], ng[:, 0], ng[:, -1]])
        if np.isin(border, self._front_states).any():
            self.front_hit_edge = True

    # ── Diagnostics ───────────────────────────────────────────────────────────

    def counts(self) -> dict:
        g = self.grid
        front_pos = np.argwhere(np.isin(g, self._front_states))
        V_mean_dist = (float(np.sqrt(((front_pos - self._center)**2).sum(axis=1)).mean())
                       if len(front_pos) > 0 else float('nan'))
        n_T = int((g == self.T).sum())
        n_A = int((g == self.A).sum())
        return dict(
            T           = n_T,
            E           = int((g == self.E).sum()),
            V1          = int((g == self.V1).sum()),
            V2          = int((g == self.V2).sum()) if self.par.two_V else 0,
            V           = int(np.isin(g, self.V).sum()),
            A           = n_A,
            DV          = int((g == self.DV).sum()),
            DN          = int((g == self.DN).sum()),
            IFN         = self.IFN,
            virions     = self.total_virions,
            V_mean_dist = V_mean_dist,
            # Canonical plaque size (SI fig. S6C): every cell that is neither T
            # nor A, i.e. has been infected at some point. Use this key rather
            # than reconstructing plaque size from state counts at the call site.
            plaque      = int(g.size - n_T - n_A),
            t           = self.t,
        )

    def render(self) -> np.ndarray:
        g   = self.grid
        img = np.ones((*g.shape, 3))
        img[g == self.E] = [1.00, 0.75, 0.00]   # gold
        if self.par.two_V:
            img[g == self.V1] = [1.00, 0.40, 0.10]  # orange-red (low shed)
            img[g == self.V2] = [0.85, 0.10, 0.10]  # crimson    (peak shed)
        else:
            img[g == self.V1] = [0.85, 0.10, 0.10]  # red
        img[g == self.A]  = [0.10, 0.75, 0.20]
        img[g == self.DV] = [0.25, 0.20, 0.20]
        img[g == self.DN] = [0.60, 0.60, 0.65]
        return img

In [ ]:
# ── Parallel ensemble runner ─────────────────────────────────────────────────
# Runs multiple independent stochastic realizations across CPU cores. Uses
# `multiprocess` (dill-based) rather than stdlib multiprocessing/concurrent.futures
# because CellularAutomaton/Params are defined here in the notebook (__main__),
# and stdlib pickle can't ship those to worker processes on macOS's spawn start method.

def _run_ensemble_member(par, n_steps):
    ca_r = CellularAutomaton(par, seed=None)
    ca_r.seed_infection(n=1)
    hist_r = []
    for _ in range(n_steps):
        ca_r.step()
        c = ca_r.counts()
        hist_r.append(c)
        if ca_r.front_hit_edge:
            break
        if c['E'] + c['V'] == 0:
            # Infection died out: freeze cell counts and let IFN decay to zero,
            # then pad out to n_steps. Without this, the shortest (fastest-
            # extinguishing) run would drag the whole ensemble's histories
            # down via the min_len truncation below.
            frozen = dict(c, IFN=0.0)
            for k in range(1, n_steps - len(hist_r) + 1):
                hist_r.append(dict(frozen, t=c['t'] + k * par.dt))
            break

    a_r   = np.array([c['t']       for c in hist_r])
    vir_r = np.array([c['virions'] for c in hist_r])
    ifn_r = np.array([c['IFN']     for c in hist_r])
    v_r   = np.array([c['V']       for c in hist_r])

    idx_ifn = np.argmax(ifn_r > 0)
    last    = hist_r[-1]

    return dict(
        a            = a_r,
        rate         = np.diff(vir_r) / np.diff(a_r),
        plaque       = np.array([c['plaque'] for c in hist_r]),
        v            = v_r,
        a_first_ifn  = a_r[idx_ifn] if ifn_r[idx_ifn] > 0 else np.nan,
        hit_edge     = ca_r.front_hit_edge,
        still_active = (last['E'] + last['V']) > 0,
    )


def run_ensemble(par, n_steps, n_runs, desc, processes=None):
    """Run n_runs independent realizations of `par` in parallel and collect summary arrays.

    Runs that hit the grid edge stop early and are genuinely shorter (data beyond
    that point is invalid due to periodic-boundary wraparound). Runs whose infection
    dies out are instead padded (frozen counts, IFN decayed to 0) back up to n_steps,
    so they don't truncate the rest of the ensemble. Every run's arrays are truncated
    to the shortest run (normally an edge-hit run, if any) so they can be stacked
    uniformly.

    CAUTION — this common truncation applies to the whole ensemble, so a single run
    that hits the edge early cuts the Lotka-Euler integral short for *every* run and
    biases the resulting q estimate upward. If the edge-hit warning below fires,
    enlarge L and re-run rather than accepting the q value.
    """
    worker = functools.partial(_run_ensemble_member, n_steps=n_steps)
    with mp.Pool(processes or mp.cpu_count()) as pool:
        results = list(tqdm(pool.imap(worker, [par] * n_runs), total=n_runs, desc=desc))

    n_hit_edge = sum(r['hit_edge'] for r in results)
    if n_hit_edge:
        warnings.warn(
            f"{n_hit_edge}/{n_runs} runs hit the grid edge and were stopped early. "
            f"The whole ensemble is truncated to the shortest run, so q is biased high."
        )

    n_still_active = sum(r['still_active'] for r in results)
    if n_still_active:
        warnings.warn(
            f"{n_still_active}/{n_runs} runs still had active infected cells "
            f"(E/V) when the simulation ended — n_steps may be too small."
        )

    min_len = min(len(r['plaque']) for r in results)
    return dict(
        rate_histories   = [r['rate'][:min_len - 1] for r in results],
        plaque_histories = [r['plaque'][:min_len]    for r in results],
        v_histories      = [r['v'][:min_len]         for r in results],
        a_first_ifn_list = [r['a_first_ifn'] for r in results],
        a                = results[0]['a'][:min_len],
    )

# IAV

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
mode = "basic"   # "basic" | "step_IFN" | "no_IFN" | "strong_IFN"

different_beta = False
if different_beta:
    beta  = 200
    sigma = Params().sigma * 0.75
else:
    beta  = 100
    sigma = Params().sigma

par = Params(mode=mode, L=200, beta=beta, sigma=sigma)

ca = CellularAutomaton(par, seed=None)
ca.seed_infection(n=1)

n_steps        = 2500
snapshot_every = 10
snapshots, snap_ages, history = [], [], []

for i in range(n_steps):
    ca.step()
    c = ca.counts()
    history.append(c)
    if ca.front_hit_edge:
        warnings.warn(f"Front reached grid edge at t={ca.t:.1f} h (step {i}) — stopping simulation early.")
        break
    if c['E'] + c['V'] == 0:
        break
    if i % snapshot_every == 0:
        snapshots.append(ca.render())
        snap_ages.append(ca.t)

print(f"Done. a = {ca.t:.1f} h  |  {len(snapshots)} frames  |  final counts: {ca.counts()}")


In [ ]:
# ── Time series ───────────────────────────────────────────────────────────────
a_arr   = np.array([c['t']      for c in history])
T_arr   = np.array([c['T']      for c in history])
E_arr   = np.array([c['E']      for c in history])
V_arr   = np.array([c['V']      for c in history])
A_arr   = np.array([c['A']      for c in history])
DV_arr  = np.array([c['DV']     for c in history])
DN_arr  = np.array([c['DN']     for c in history])
IFN_arr = np.array([c['IFN']    for c in history])
plq_arr = np.array([c['plaque'] for c in history])

first_ifn_idx = np.argmax(IFN_arr > 0)
a_first_ifn   = a_arr[first_ifn_idx] if IFN_arr[first_ifn_idx] > 0 else None

def add_vlines(ax, legend=False):
    if a_first_ifn is not None:
        ax.axvline(a_first_ifn, color='mediumpurple', ls='--', lw=1.2,
                   label='first IFN' if legend else None)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(11, 9), sharex=True)

# ── Dominant populations ──────────────────────────────────────────────────────
ax1.plot(a_arr, T_arr+A_arr, color='black',            label='Live cells (T + A)')
ax1.plot(a_arr, T_arr,       color='black', ls='--',   label='T')
ax1.plot(a_arr, A_arr,       color='black', ls='dotted',label='A')
add_vlines(ax1, legend=True)
ax1.set_ylabel('Cell count')
ax1.legend(ncol=5, fontsize=9)
ax1.set_title('Dominant populations')

# ── Dead populations ──────────────────────────────────────────────────────────
ax2.plot(a_arr, DV_arr+DN_arr, color='forestgreen',            label='Dead cells')
ax2.plot(a_arr, DV_arr,        color='forestgreen', ls='--',   label='D_V')
ax2.plot(a_arr, DN_arr,        color='forestgreen', ls='dotted',label='D_N')
add_vlines(ax2)
ax2.set_ylabel('Cell count')
ax2.legend(ncol=3, fontsize=9)
ax2.set_title('Dead populations')

# ── Transient states ──────────────────────────────────────────────────────────
# Plaque size is the canonical definition from counts(): all cells that are
# neither T nor A, i.e. have been infected at some point.
ax3.plot(a_arr, E_arr,   color='goldenrod', label='E')
ax3.plot(a_arr, V_arr,   color='crimson',   label='V')
ax3.plot(a_arr, plq_arr, color='red', ls='--', label='Plaque size')
add_vlines(ax3)
ax3.set_ylabel('Cell count')
ax3.legend(ncol=4, fontsize=9)
ax3.set_title('Transient states')

# ── IFN ───────────────────────────────────────────────────────────────────────
ax4.plot(a_arr, IFN_arr, color='mediumpurple')
add_vlines(ax4)
ax4.set_ylabel('IFN (a.u.)')
ax4.set_xlabel('Age (hours)')
ax4.set_title('Global IFN')

plt.tight_layout()
plt.show()

In [ ]:
# ── IAV: all analytical plots ─────────────────────────────────────────────────
lam = 0.25   # discount rate (h⁻¹)

a_arr      = np.array([c['t']           for c in history])
virion_arr = np.array([c['virions']     for c in history])
vdist      = np.array([c['V_mean_dist'] for c in history])

da_steps   = np.diff(a_arr)
a_mid      = 0.5 * (a_arr[:-1] + a_arr[1:])
rate       = np.diff(virion_arr) / da_steps
discounted = rate * np.exp(-lam * a_mid)
integral   = np.cumsum(discounted * da_steps)
q          = par.beta / integral[-1]

b_disc     = (rate / par.beta) * q * np.exp(-lam * a_mid)   # discounted kernel b(a)·e^{-λa}
le_running = np.cumsum(b_disc * da_steps)

a_ref    = 1 / par.r_v
ref_line = np.where(a_arr >= a_ref, a_arr - a_ref, np.nan)

axissize = 16
ticksize = 15
legendsize = 14

fig, axes = plt.subplots(5, 1, figsize=(6.5, 16), sharex=False)

axes[0].plot(a_arr, vdist,    color='crimson', label='Front')
axes[0].plot(a_arr, ref_line, 'k--',           label='Reference: 1 cell/h')
add_vlines(axes[0])
axes[0].set_ylabel('Mean dist. ' + r"[$d_c$]", fontsize=axissize)
axes[0].legend(fontsize=legendsize,loc = "lower right")
axes[0].set_xticks([],[])
axes[0].tick_params(labelsize=ticksize)

axes[1].plot(a_mid, rate, color='crimson')
add_vlines(axes[1])
axes[1].set_ylabel('Shedding rate' + r" [h$^{-1}$]", fontsize=axissize)
axes[1].set_xlabel('Age (h)', fontsize=axissize)
axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
axes[1].tick_params(labelsize=ticksize)
axes[1].yaxis.get_offset_text().set_fontsize(ticksize)

axes[2].plot(a_arr, virion_arr, color='crimson')
add_vlines(axes[2])
axes[2].set_ylabel('Cumulative virions', fontsize=axissize)
axes[2].set_xlabel('Age (h)', fontsize=axissize)
axes[2].set_title('Total virus produced', fontsize=axissize)
axes[2].tick_params(labelsize=ticksize)

axes[3].plot(a_mid, b_disc, color='steelblue')
add_vlines(axes[3])
axes[3].set_ylabel(r'$b(a)\cdot e^{-\lambda a}$', fontsize=axissize)
axes[3].set_xlabel('Age (h)', fontsize=axissize)
axes[3].set_title(fr'Discounted kernel  [q = {q:.4f}]', fontsize=axissize)
axes[3].tick_params(labelsize=ticksize)

axes[4].plot(a_mid, le_running, color='steelblue',
             label=r'$\int_0^a b(a)\,e^{-\lambda a}\,da$')
axes[4].axhline(1, color='black', ls='--', lw=1, label='1 (LE condition)')
add_vlines(axes[4])
axes[4].set_ylabel('Running LE integral', fontsize=axissize)
axes[4].set_xlabel('Age (h)', fontsize=axissize)
axes[4].set_title('Lotka-Euler integral', fontsize=axissize)
axes[4].legend(fontsize=axissize)
axes[4].tick_params(labelsize=ticksize)

plt.tight_layout(); plt.show()

In [ ]:
# ── Animation ─────────────────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color='white',            label=r'$T$  (target)',      ec='lightgray'),
    mpatches.Patch(color=[1, 0.75, 0],       label=r'$E$'),
    mpatches.Patch(color=[0.85, 0.1, 0.1],   label=r'$V$  (producer)'),
    mpatches.Patch(color=[0.1, 0.75, 0.2],   label=r'$A$  (antiviral)'),
    mpatches.Patch(color=[0.25, 0.2, 0.2],   label=r'$D_V$ (died viral)'),
    mpatches.Patch(color=[0.6, 0.6, 0.65],   label=r'$D_N$ (died IFN)'),
]

legendsize = 12
fig, ax = plt.subplots(figsize=(6, 6))
im    = ax.imshow(snapshots[0], origin='lower', interpolation='nearest')
title = ax.set_title(f'a = {snap_ages[0]:.1f} h', fontsize=12)
ax.axis('off')

L_grid = snapshots[0].shape[0]
ax.add_patch(mpatches.Rectangle((-0.5, -0.5), L_grid, L_grid,
                                 fill=False, edgecolor='black', linewidth=1.5, clip_on=False))

ax.legend(handles=legend_patches, loc='lower right', fontsize=legendsize, framealpha=0.75)

def update(frame):
    im.set_data(snapshots[frame])
    title.set_text(f'a = {snap_ages[frame]:.0f} h')
    return im, title

anim = FuncAnimation(fig, update, frames=len(snapshots), interval=20, blit=True)
plt.close()
HTML(anim.to_jshtml())


In [ ]:
n_runs = 500

print("Parameters:\n" + "\n".join(f"  {f.name} = {getattr(par, f.name)}" for f in dataclasses.fields(par)))

result = run_ensemble(par, n_steps, n_runs, desc='IAV simulations')

rate_histories   = result['rate_histories']
plaque_histories = result['plaque_histories']
a_first_ifn_list = result['a_first_ifn_list']
a_r              = result['a']

# age grid is identical across runs
a_mid_r = 0.5 * (a_r[:-1] + a_r[1:])
da_r    = np.diff(a_r)

mean_a_ifn = np.nanmean(a_first_ifn_list)


In [ ]:
# ── IAV: F(a) ensemble + q from mean / median ─────────────────────────────────
rates       = np.array(rate_histories)
mean_rate   = rates.mean(axis=0)
median_rate = np.median(rates, axis=0)

disc       = np.exp(-lam * a_mid_r)
env_mean   = mean_rate   * disc
env_median = median_rate * disc
q_mean     = par.beta / np.sum(env_mean   * da_r)
q_median   = par.beta / np.sum(env_median * da_r)

plaques     = np.array(plaque_histories)
mean_plaque = plaques.mean(axis=0)
med_plaque  = np.median(plaques, axis=0)

show_plaque_size = False
show_median      = False
legendsize      = 20
labelsize       = 20
ticklabelsize   = 16
alpha           = 0.02

def add_ensemble_vlines(ax):
    ax.axvline(mean_a_ifn, color='mediumpurple', ls='--', lw=1.2, label=fr'Mean IFN onset  ({mean_a_ifn:.1f} h)')

if show_plaque_size:
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

for i, rate_r in enumerate(rate_histories):
    ax1.plot(a_mid_r, rate_r,                            color='steelblue', alpha=alpha, lw=0.8)
    ax2.plot(a_mid_r, rate_r * disc * q_mean / par.beta, color='steelblue', alpha=alpha, lw=0.8)
    if show_plaque_size:
        ax3.plot(a_r, plaques[i], color='steelblue', alpha=alpha, lw=0.8)

ax1.plot(a_mid_r, mean_rate,                       color='crimson',    lw=2, label=fr'Mean   —   $q$ = {q_mean:.2f}')
ax2.plot(a_mid_r, env_mean * q_mean / par.beta,    color='crimson',    lw=2, label=fr'Mean   —   $q$ = {q_mean:.2f}')
if show_median:
    ax1.plot(a_mid_r, median_rate,                       color='darkorange', lw=2, label=fr'Median —   $q$ = {q_median:.2f}')
    ax2.plot(a_mid_r, env_median * q_median / par.beta,  color='darkorange', lw=2, label='Median')
if show_plaque_size:
    ax3.plot(a_r, mean_plaque, color='crimson',    lw=2, label='Mean')
    if show_median:
        ax3.plot(a_r, med_plaque, color='darkorange', lw=2, label='Median')

add_ensemble_vlines(ax1)
add_ensemble_vlines(ax2)
if show_plaque_size:
    add_ensemble_vlines(ax3)

ax1.set_ylabel(r'Shedding rate [h$^{-1}$]', fontsize=labelsize)
ax2.set_ylabel('Discounted kernel', fontsize=labelsize);   ax2.legend(fontsize=legendsize)
ax1.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax1.tick_params(labelsize=ticklabelsize)
ax2.tick_params(labelsize=ticklabelsize)
ax1.yaxis.get_offset_text().set_fontsize(ticklabelsize)

if show_plaque_size:
    ax3.set_ylabel('Cells', fontsize=labelsize);      ax3.legend(fontsize=legendsize)
    ax3.set_xlabel('Age (h)', fontsize=labelsize);    ax3.set_title('Plaque size (all cells except T or A)')
    ax3.tick_params(labelsize=ticklabelsize)
else:
    ax2.set_xlabel('Age (h)', fontsize=labelsize)

print(fr'IAV: {n_runs} runs,  beta={par.beta}')
plt.tight_layout()
plt.xlim(0,72)
plt.show()

# ── q convergence over ensemble size ──────────────────────────────────────────
q_running = np.array([
    par.beta / np.sum(rates[:k].mean(axis=0) * disc * da_r)
    for k in range(1, n_runs + 1)
])

fig2, ax_q = plt.subplots(figsize=(7, 3.5))
ax_q.plot(range(1, n_runs + 1), q_running, color='steelblue', lw=2)
ax_q.axhline(q_mean, color='crimson', ls='--', lw=1.5, label=fr'Final  $q$ = {q_mean:.4f}')
ax_q.set_xlabel('Number of runs', fontsize=labelsize)
ax_q.set_ylabel(r'$q$ estimate', fontsize=labelsize)
ax_q.legend(fontsize=legendsize)
ax_q.tick_params(labelsize=ticklabelsize)
plt.tight_layout()
plt.show()

# SARS-CoV-2 parametrisation

Two-phase infectious period: **V1** (low shedding) → **V2** (peak shedding) → D_V.
beta = 500, V_ratio = 100, other parameters at default values.

Note that the burst is split so the *total* expected yield is exactly beta = 500,
giving 4.95 virions from V1 and 495.0 from V2 (not 5 and 500).

In [ ]:
@dataclass
class ParamsSC2(Params):
    """SARS-CoV-2 parameter defaults — overrides the fields that differ from IAV."""
    virus:  str   = "SARS_CoV2"
    beta:   float = 500.0
    sigma:  float = 1.75
    two_V:  bool  = True
    p_0:    float = 1e-4    # 50x lower than IAV
    p_max:  float = 5e-2    # same as IAV
    L:      int   = 500

In [ ]:
# ── SARS-CoV-2 run ────────────────────────────────────────────────────────────
mode_sc = "basic"   # "basic" | "step_IFN" | "no_IFN" | "strong_IFN"
par_sc  = ParamsSC2(mode=mode_sc)

ca_sc = CellularAutomaton(par_sc, seed=None)
print(f"r1 = {ca_sc._r1:.3f} virions/h  (V1 shedding,  tau_v  = {par_sc.tau_v} h)")
print(f"r2 = {ca_sc._r2:.2f} virions/h  (V2 shedding,  tau_v2 = {par_sc.tau_v2} h)")
print(f"Expected burst: r1*tau_v + r2*tau_v2 = {ca_sc._r1*par_sc.tau_v + ca_sc._r2*par_sc.tau_v2:.1f}  (should equal beta={par_sc.beta})")
print(f"  split as {ca_sc._r1*par_sc.tau_v:.2f} from V1 and {ca_sc._r2*par_sc.tau_v2:.1f} from V2")

ca_sc.seed_infection(n=1)

n_steps_sc     = 3500
snapshot_every = 10
snapshots_sc, snap_ages_sc, history_sc = [], [], []

for i in range(n_steps_sc):
    ca_sc.step()
    c = ca_sc.counts()
    history_sc.append(c)
    if ca_sc.front_hit_edge:
        warnings.warn(f"Front reached grid edge at t={ca_sc.t:.1f} h (step {i}) — stopping simulation early.")
        break
    if c['E'] + c['V'] == 0:
        break
    if i % snapshot_every == 0:
        snapshots_sc.append(ca_sc.render())
        snap_ages_sc.append(ca_sc.t)

print(f"\nDone. a = {ca_sc.t:.1f} h  |  final counts: {ca_sc.counts()}")


In [ ]:
# ── Time series ───────────────────────────────────────────────────────────────
a_sc   = np.array([c['t']      for c in history_sc])
T_sc   = np.array([c['T']      for c in history_sc])
E_sc   = np.array([c['E']      for c in history_sc])
V1_sc  = np.array([c['V1']     for c in history_sc])
V2_sc  = np.array([c['V2']     for c in history_sc])
A_sc   = np.array([c['A']      for c in history_sc])
DV_sc  = np.array([c['DV']     for c in history_sc])
DN_sc  = np.array([c['DN']     for c in history_sc])
IFN_sc = np.array([c['IFN']    for c in history_sc])
plq_sc = np.array([c['plaque'] for c in history_sc])

a_fifn_sc = a_sc[np.argmax(IFN_sc > 0)] if (IFN_sc > 0).any() else None

def add_vlines_sc(ax, legend=False):
    if a_fifn_sc is not None:
        ax.axvline(a_fifn_sc, color='mediumpurple', ls='--', lw=1.2,
                   label='first IFN' if legend else None)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(11, 9), sharex=True)

ax1.plot(a_sc, T_sc+A_sc, 'k',        label='T+A')
ax1.plot(a_sc, T_sc,      'k--',      label='T')
ax1.plot(a_sc, A_sc,      'k:',       label='A')
add_vlines_sc(ax1, legend=True)
ax1.set_ylabel('Cell count'); ax1.legend(ncol=5, fontsize=9); ax1.set_title('Dominant populations')

ax2.plot(a_sc, DV_sc+DN_sc, color='forestgreen',        label='Dead')
ax2.plot(a_sc, DV_sc,       color='forestgreen', ls='--', label='D_V')
ax2.plot(a_sc, DN_sc,       color='forestgreen', ls=':',  label='D_N')
add_vlines_sc(ax2)
ax2.set_ylabel('Cell count'); ax2.legend(ncol=3, fontsize=9); ax2.set_title('Dead populations')

# Plaque size uses the canonical definition from counts(): not T and not A.
ax3.plot(a_sc, E_sc,   color='goldenrod',  label='E')
ax3.plot(a_sc, V1_sc,  color=[1,.4,.1],    label='V1 (low shed)')
ax3.plot(a_sc, V2_sc,  color='crimson',    label='V2 (peak shed)')
ax3.plot(a_sc, plq_sc, 'r--',              label='Plaque')
add_vlines_sc(ax3)
ax3.set_ylabel('Cell count'); ax3.legend(ncol=5, fontsize=9); ax3.set_title('Transient states')

ax4.plot(a_sc, IFN_sc, color='mediumpurple')
add_vlines_sc(ax4)
ax4.set_ylabel('IFN (a.u.)'); ax4.set_xlabel('Age (hours)'); ax4.set_title('Global IFN')

plt.tight_layout(); plt.show()

In [ ]:
# ── SARS-CoV-2: all analytical plots ─────────────────────────────────────────
vdist_sc = np.array([c['V_mean_dist'] for c in history_sc])
a_ref_sc = 1 / par_sc.r_v
ref_sc   = np.where(a_sc >= a_ref_sc, a_sc - a_ref_sc, np.nan)

vir_sc    = np.array([c['virions'] for c in history_sc])
da_sc     = np.diff(a_sc)
a_mid_sc  = 0.5 * (a_sc[:-1] + a_sc[1:])
rate_sc   = np.diff(vir_sc) / da_sc
disc_sc   = rate_sc * np.exp(-lam * a_mid_sc)
integ_sc  = np.cumsum(disc_sc * da_sc)
q_sc      = par_sc.beta / integ_sc[-1]
b_disc_sc = (rate_sc / par_sc.beta) * q_sc * np.exp(-lam * a_mid_sc)
le_sc     = np.cumsum(b_disc_sc * da_sc)

fig, axes = plt.subplots(5, 1, figsize=(11, 16), sharex=False)

axes[0].plot(a_sc, vdist_sc, color='crimson', label='Mean E+V distance')
axes[0].plot(a_sc, ref_sc,   'k--',           label='Reference: 1 cell/h')
add_vlines_sc(axes[0]); axes[0].set_ylabel('Mean dist (cells)')
axes[0].set_xlabel('Age (h)'); axes[0].legend(); axes[0].set_title('Front expansion speed')

axes[1].plot(a_sc, vir_sc, color='crimson')
add_vlines_sc(axes[1]); axes[1].set_ylabel('Cumulative virions')
axes[1].set_xlabel('Age (h)'); axes[1].set_title('Total virus produced')

axes[2].plot(a_mid_sc, rate_sc, color='crimson')
add_vlines_sc(axes[2]); axes[2].set_ylabel('Virions / h')
axes[2].set_xlabel('Age (h)'); axes[2].set_title(r'Virus production rate $F(a)$')

axes[3].plot(a_mid_sc, b_disc_sc, color='steelblue')
add_vlines_sc(axes[3]); axes[3].set_ylabel(r'$b(a)\cdot e^{-\lambda a}$')
axes[3].set_xlabel('Age (h)'); axes[3].set_title(fr'Discounted kernel  [q = {q_sc:.4f}]')

axes[4].plot(a_mid_sc, le_sc, color='steelblue',
             label=r'$\int_0^a b(a)\,e^{-\lambda a}\,da$')
axes[4].axhline(1, color='black', ls='--', lw=1, label='1 (LE condition)')
add_vlines_sc(axes[4]); axes[4].set_ylabel('Running LE integral')
axes[4].set_xlabel('Age (h)')
axes[4].set_title('Lotka-Euler integral'); axes[4].legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── Animation ─────────────────────────────────────────────────────────────────
legend_patches_sc = [
    mpatches.Patch(color='white',            label='T  (target)',      ec='lightgray'),
    mpatches.Patch(color=[1, 0.75, 0],       label='E'),
    mpatches.Patch(color=[1, 0.4, 0.1],      label='V1 (low shed)'),
    mpatches.Patch(color=[0.85, 0.1, 0.1],   label='V2 (peak shed)'),
    mpatches.Patch(color=[0.1, 0.75, 0.2],   label='A  (antiviral)'),
    mpatches.Patch(color=[0.25, 0.2, 0.2],   label='D_V (died viral)'),
    mpatches.Patch(color=[0.6, 0.6, 0.65],   label='D_N (died IFN)'),
]

fig, ax = plt.subplots(figsize=(6, 6))
im    = ax.imshow(snapshots_sc[0], origin='lower', interpolation='nearest')
title = ax.set_title(f'a = {snap_ages_sc[0]:.1f} h', fontsize=12)
ax.axis('off')
ax.legend(handles=legend_patches_sc, loc='lower right', fontsize=7, framealpha=0.85)

def update_sc(frame):
    im.set_data(snapshots_sc[frame])
    title.set_text(f'a = {snap_ages_sc[frame]:.1f} h')
    return im, title

anim_sc = FuncAnimation(fig, update_sc, frames=len(snapshots_sc), interval=20, blit=True)
plt.close()
HTML(anim_sc.to_jshtml())

In [ ]:
n_runs_sc = 200

result_sc = run_ensemble(par_sc, n_steps_sc, n_runs_sc, desc='SARS-CoV-2 simulations')

rate_histories_sc   = result_sc['rate_histories']
plaque_histories_sc = result_sc['plaque_histories']
v_histories_sc      = result_sc['v_histories']
a_first_ifn_list_sc = result_sc['a_first_ifn_list']
a_r                 = result_sc['a']

# age grid is identical across runs
a_mid_sc_r = 0.5 * (a_r[:-1] + a_r[1:])
da_sc_r    = np.diff(a_r)

mean_a_ifn_sc = np.nanmean(a_first_ifn_list_sc)

In [ ]:
print("Parameters:\n" + "\n".join(f"  {f.name} = {getattr(par_sc, f.name)}" for f in dataclasses.fields(par_sc)))

In [ ]:
# ── plot ──────────────────────────────────────────────────────────────────────
rates_sc       = np.array(rate_histories_sc)
mean_rate_sc   = rates_sc.mean(axis=0)
median_rate_sc = np.median(rates_sc, axis=0)

disc_sc_r       = np.exp(-lam * a_mid_sc_r)
env_mean_sc     = mean_rate_sc   * disc_sc_r
env_median_sc   = median_rate_sc * disc_sc_r
q_mean_sc       = par_sc.beta / np.sum(env_mean_sc   * da_sc_r)
q_median_sc     = par_sc.beta / np.sum(env_median_sc * da_sc_r)

plaques_sc     = np.array(plaque_histories_sc)
mean_plaque_sc = plaques_sc.mean(axis=0)
med_plaque_sc  = np.median(plaques_sc, axis=0)

vs_sc          = np.array(v_histories_sc)
mean_v_sc      = vs_sc.mean(axis=0)
med_v_sc       = np.median(vs_sc, axis=0)

show_plaque_size    = True
show_median         = False
show_infected       = False
legendsize          = 20
labelsize           = 20
ticklabelsize       = 16
alpha               = 0.08

def add_ensemble_vlines_sc(ax):
    ax.axvline(mean_a_ifn_sc, color='mediumpurple', ls='--', lw=1.2, label=fr'Mean IFN onset  ({mean_a_ifn_sc:.1f} h)')

n_panels = 2 + show_infected + show_plaque_size
fig, axes = plt.subplots(n_panels, 1, figsize=(10, 5 * n_panels), sharex=True)
ax1, ax2 = axes[0], axes[1]
ax_idx = 2
if show_infected:
    ax_inf = axes[ax_idx]; ax_idx += 1
if show_plaque_size:
    ax_plq = axes[ax_idx]; ax_idx += 1

for i, rate_r in enumerate(rate_histories_sc):
    ax1.plot(a_mid_sc_r, rate_r,                                       color='steelblue', alpha=alpha, lw=0.8)
    ax2.plot(a_mid_sc_r, rate_r * disc_sc_r * q_mean_sc / par_sc.beta, color='steelblue', alpha=alpha, lw=0.8)
    if show_infected:
        ax_inf.plot(a_r, vs_sc[i], color='steelblue', alpha=alpha, lw=0.8)
    if show_plaque_size:
        ax_plq.plot(a_r, plaques_sc[i], color='steelblue', alpha=alpha, lw=0.8)

ax1.plot(a_mid_sc_r, mean_rate_sc,                            color='crimson',    lw=2, label=fr'Mean   —   $q$ = {q_mean_sc:.2f}')
ax2.plot(a_mid_sc_r, env_mean_sc * q_mean_sc / par_sc.beta,   color='crimson',    lw=2, label=fr'Mean   —   $q$ = {q_mean_sc:.2f}')
if show_median:
    ax1.plot(a_mid_sc_r, median_rate_sc,                             color='darkorange', lw=2, label=fr'Median —   $q$ = {q_median_sc:.2f}')
    ax2.plot(a_mid_sc_r, env_median_sc * q_median_sc / par_sc.beta,  color='darkorange', lw=2, label='Median')
if show_infected:
    ax_inf.plot(a_r, mean_v_sc, color='crimson',    lw=2, label='Mean')
    if show_median:
        ax_inf.plot(a_r, med_v_sc, color='darkorange', lw=2, label='Median')
if show_plaque_size:
    ax_plq.plot(a_r, mean_plaque_sc, color='crimson',    lw=2, label='Mean')
    if show_median:
        ax_plq.plot(a_r, med_plaque_sc, color='darkorange', lw=2, label='Median')

add_ensemble_vlines_sc(ax1)
add_ensemble_vlines_sc(ax2)
if show_infected:
    add_ensemble_vlines_sc(ax_inf)
if show_plaque_size:
    add_ensemble_vlines_sc(ax_plq)

ax1.set_ylabel('Shedding rate' + r' [h$^{-1}$]', fontsize=labelsize)
ax2.set_ylabel('Discounted shedding rate' + r' [h$^{-1}$]', fontsize=labelsize);   ax2.legend(fontsize=legendsize)
ax1.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax1.tick_params(labelsize=ticklabelsize)
ax2.tick_params(labelsize=ticklabelsize)
ax1.yaxis.get_offset_text().set_fontsize(ticklabelsize)

if show_infected:
    ax_inf.set_ylabel('Productively infected (V1+V2)', fontsize=labelsize)
    ax_inf.legend(fontsize=legendsize)
    ax_inf.tick_params(labelsize=ticklabelsize)
if show_plaque_size:
    ax_plq.set_ylabel('Plaque size [cells]', fontsize=labelsize)
    ax_plq.set_xlabel('Age (h)', fontsize=labelsize)
    ax_plq.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    ax_plq.tick_params(labelsize=ticklabelsize)
    ax_plq.yaxis.get_offset_text().set_fontsize(ticklabelsize)
elif show_infected:
    ax_inf.set_xlabel('Age (h)', fontsize=labelsize)
else:
    ax2.set_xlabel('Age (h)', fontsize=labelsize)

print(fr'SARS-CoV-2: {n_runs_sc} runs,  beta={par_sc.beta},  two_V=True')
plt.tight_layout()
plt.xlim(0,200)
plt.show()

# ── q convergence over ensemble size ──────────────────────────────────────────
q_running_sc = np.array([
    par_sc.beta / np.sum(rates_sc[:k].mean(axis=0) * disc_sc_r * da_sc_r)
    for k in range(1, n_runs_sc + 1)
])

fig2, ax_q = plt.subplots(figsize=(7, 3.5))
ax_q.plot(range(1, n_runs_sc + 1), q_running_sc, color='steelblue', lw=2)
ax_q.axhline(q_mean_sc, color='crimson', ls='--', lw=1.5, label=fr'Final  $q$ = {q_mean_sc:.2f}')
ax_q.set_xlabel('Number of runs', fontsize=labelsize)
ax_q.set_ylabel(r'$q$ estimate', fontsize=labelsize)
ax_q.legend(fontsize=legendsize)
ax_q.tick_params(labelsize=ticklabelsize)
plt.tight_layout()
plt.show()
